In [1]:
# ============================================================
# CELL 1: INSTALL ALL DEPENDENCIES (Run once, restart kernel after)
# ============================================================
# Core ML
%pip install -q numpy pandas opencv-python-headless Pillow tqdm scipy PyWavelets
%pip install -q matplotlib seaborn scikit-learn xgboost lightgbm joblib ipywidgets

# PyTorch (CUDA 11.8 compatible)
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

# Vision models
%pip install -q git+https://github.com/openai/CLIP.git
%pip install -q timm albumentations

# SigLIP + DIRE dependencies (FIXED)
%pip install -q transformers safetensors diffusers accelerate

# Upgrade typing for timm compatibility
%pip install -q --upgrade typing_extensions

%pip install sentencepiece protobuf

print("All packages installed. Restart kernel if this is first run.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
All packages installed. Restart kernel if this is first run.


In [1]:
# ============================================================
# CELL 2: IMPORTS, CONFIG, SEEDS, GPU
# ============================================================
import os, sys, warnings, io, random, copy, gc
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
from scipy.fft import fft2, fftshift
import pywt

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (f1_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.base import BaseEstimator, ClassifierMixin
import xgboost as xgb
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A

warnings.filterwarnings('ignore')
#data/DCU 2026 ML challenge - external 2

# Use ".." to go to the parent folder where the dataset lives
BASE_PATH = Path("../DCU 2026 ML challenge - external 2")

IMAGE_DIR = BASE_PATH / "images/images_final_sample"
TRAIN_CSV = BASE_PATH / "train.csv"
TEST_CSV  = BASE_PATH / "test.csv"

print("--- PATH VERIFICATION ---")
print("BASE_PATH:", BASE_PATH)
print("IMAGE_DIR exists?", IMAGE_DIR.exists())
print("TRAIN_CSV exists?", TRAIN_CSV.exists())
print("TEST_CSV exists?", TEST_CSV.exists())
print("-------------------------")

# ─── Config ──────────────────────────────────────────────
SEED         = 42
CACHE_DIR    = Path("./feature_cache_v7")
MODEL_DIR    = Path("./saved_models_v7")
CACHE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

N_FOLDS      = 5
FORCE_FRESH  = True
SIGLIP_DIM   = 1152      # SigLIP So400m output dimension
DINO_L_DIM   = 1024      # DINOv2-Large output dimension
CNN_DIM      = 1280      # EfficientNet-B0
FORENSIC_DIM = 114       # ELA + FFT + Noise (from v5)
N_TTA        = 5

# ─── Reproducibility ─────────────────────────────────────
def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seeds()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Config: N_FOLDS={N_FOLDS}, N_TTA={N_TTA}")
print(f"Models: SigLIP({SIGLIP_DIM}d) + DINOv2-L({DINO_L_DIM}d) + DIRE + CNN({CNN_DIM}d) + Forensic({FORENSIC_DIM}d)")

--- PATH VERIFICATION ---
BASE_PATH: ../DCU 2026 ML challenge - external 2
IMAGE_DIR exists? True
TRAIN_CSV exists? True
TEST_CSV exists? True
-------------------------
Device: cuda
GPU: NVIDIA GeForce RTX 3090
VRAM: 25.4 GB
Config: N_FOLDS=5, N_TTA=5
Models: SigLIP(1152d) + DINOv2-L(1024d) + DIRE + CNN(1280d) + Forensic(114d)


In [2]:
""" Only for paths.
import os
from pathlib import Path

print(f"Current folder: {os.getcwd()}")
print("\nItems in current folder:", os.listdir('.'))
print("\nItems in parent folder (..):", os.listdir('..') if os.path.exists('..') else "N/A")

# Try to find the data folder by walking up
target = "data"
p = Path.cwd()
for i in range(3): # Check up to 3 levels up
    if (p / target).exists():
        print(f"\n✅ Found 'data' at: {(p / target).resolve()}")
        break
    p = p.parent
else:
    print("\n❌ Could not find 'data' folder in parent directories.")
"""

' Only for paths.\nimport os\nfrom pathlib import Path\n\nprint(f"Current folder: {os.getcwd()}")\nprint("\nItems in current folder:", os.listdir(\'.\'))\nprint("\nItems in parent folder (..):", os.listdir(\'..\') if os.path.exists(\'..\') else "N/A")\n\n# Try to find the data folder by walking up\ntarget = "data"\np = Path.cwd()\nfor i in range(3): # Check up to 3 levels up\n    if (p / target).exists():\n        print(f"\n✅ Found \'data\' at: {(p / target).resolve()}")\n        break\n    p = p.parent\nelse:\n    print("\n❌ Could not find \'data\' folder in parent directories.")\n'

In [3]:
# ============================================================
# CELL 3: DATASET LOADING
# ============================================================
df_train_raw = pd.read_csv(TRAIN_CSV)
df_test_raw  = pd.read_csv(TEST_CSV)

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Train: {df_train_raw.shape} | Columns: {df_train_raw.columns.tolist()}")
print(f"Test:  {df_test_raw.shape}  | Columns: {df_test_raw.columns.tolist()}")

n_real = (df_train_raw.ground_truth == 0).sum()
n_ai   = (df_train_raw.ground_truth == 1).sum()
print(f"Class balance: Real={n_real} ({n_real/len(df_train_raw):.1%}), AI={n_ai} ({n_ai/len(df_train_raw):.1%})")

df_train = df_train_raw.copy()
df_train['label']    = df_train['ground_truth'].astype(int)
df_train['filepath'] = df_train['image_id'].apply(lambda x: str(IMAGE_DIR / x))

df_test = df_test_raw.copy()
df_test['filepath'] = df_test['image_id'].apply(lambda x: str(IMAGE_DIR / x))

# Verify
train_ok = sum(1 for p in df_train['filepath'].head(20) if Path(p).exists())
test_ok  = sum(1 for p in df_test['filepath'].head(20) if Path(p).exists())
print(f"Path check: {train_ok}/20 train, {test_ok}/20 test images found")

train_paths = df_train['filepath'].tolist()
test_paths  = df_test['filepath'].tolist()
y_all       = df_train['ground_truth'].values

print(f"Ready: {len(train_paths)} train, {len(test_paths)} test images")

DATASET OVERVIEW
Train: (4800, 2) | Columns: ['image_id', 'ground_truth']
Test:  (2058, 1)  | Columns: ['image_id']
Class balance: Real=2485 (51.8%), AI=2315 (48.2%)
Path check: 20/20 train, 20/20 test images found
Ready: 4800 train, 2058 test images


In [4]:
# ============================================================
# CELL 4: IMAGE LOADING & AUGMENTATION UTILITIES
# ============================================================

def load_image_pil(path):
    try:
        return Image.open(str(path)).convert('RGB')
    except Exception:
        try:
            img = cv2.imread(str(path))
            if img is not None:
                return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        except Exception:
            pass
    return None

def load_image_np(path, size=(256, 256)):
    try:
        img = Image.open(str(path)).convert('RGB')
        if size:
            img = img.resize(size, Image.Resampling.LANCZOS)
        return np.array(img, dtype=np.float32)
    except Exception:
        return None

# ── Transform constants ──────────────────────────────────────
SIGLIP_MEAN = [0.5, 0.5, 0.5]
SIGLIP_STD  = [0.5, 0.5, 0.5]
DINO_MEAN   = [0.485, 0.456, 0.406]
DINO_STD    = [0.229, 0.224, 0.225]

# ── Albumentations helpers (tested API for Renku's version) ──
def get_train_transform_albu(mean, std, size=224):
    return A.Compose([
        A.RandomResizedCrop(size=(size, size), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.3),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.GaussNoise(var_limit=(5, 30), p=0.2),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3),
        A.Normalize(mean=mean, std=std),
        A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
    ])

def get_val_transform_albu(mean, std, size=224):
    return A.Compose([
        A.Resize(height=size+32, width=size+32),
        A.CenterCrop(height=size, width=size),
        A.Normalize(mean=mean, std=std),
    ])

def get_tta_transform_albu(mean, std, size=224):
    return A.Compose([
        A.RandomResizedCrop(size=(size, size), scale=(0.9, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=mean, std=std),
    ])

class AlbuDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = load_image_pil(self.paths[idx])
        if img is None:
            img = Image.new('RGB', (224, 224), (128, 128, 128))
        img_np = np.array(img)
        augmented = self.transform(image=img_np)
        img_tensor = torch.from_numpy(augmented['image'].transpose(2, 0, 1)).float()
        lbl = self.labels[idx] if self.labels is not None else -1
        return img_tensor, torch.tensor(lbl, dtype=torch.float32)

def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print("Augmentation utilities ready.")
print("  RandomResizedCrop uses size=(h,w) tuple")
print("  Resize/CenterCrop use height=h, width=w kwargs")
print("  ImageCompression(70-100) for forensic robustness")


Augmentation utilities ready.
  RandomResizedCrop uses size=(h,w) tuple
  Resize/CenterCrop use height=h, width=w kwargs
  ImageCompression(70-100) for forensic robustness


In [5]:
# ============================================================
# CELL 5: FEATURE EXTRACTION — SigLIP So400m (1152-dim)
# ============================================================
# Replaces CLIP ViT-L/14. Better calibrated sigmoid loss,
# superior for binary classification tasks.
from transformers import SiglipModel, SiglipProcessor
set_seeds()

def extract_siglip_features(image_paths, batch_size=32):
    print("Loading SigLIP So400m-patch14-224...")
    model = SiglipModel.from_pretrained("google/siglip-so400m-patch14-224")
    processor = SiglipProcessor.from_pretrained("google/siglip-so400m-patch14-224")
    model = model.to(DEVICE).eval()
    vision_model = model.vision_model

    feats_all = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='SigLIP'):
        batch_imgs = []
        for p in image_paths[i:i+batch_size]:
            img = load_image_pil(p)
            if img is None:
                img = Image.new('RGB', (224, 224), (128, 128, 128))
            batch_imgs.append(img)

        inputs = processor(images=batch_imgs, return_tensors="pt", padding=True)
        pixel_values = inputs['pixel_values'].to(DEVICE)

        with torch.no_grad():
            outputs = vision_model(pixel_values=pixel_values)
            feats = outputs.pooler_output.float()

        feats = feats / feats.norm(dim=-1, keepdim=True)
        feats_all.append(feats.cpu().numpy())

    del model, vision_model, processor
    torch.cuda.empty_cache()
    gc.collect()
    return np.vstack(feats_all).astype(np.float32)

cache = CACHE_DIR
if FORCE_FRESH or not (cache / 'siglip_train.npy').exists():
    siglip_train = extract_siglip_features(train_paths)
    siglip_test  = extract_siglip_features(test_paths)
    np.save(cache / 'siglip_train.npy', siglip_train)
    np.save(cache / 'siglip_test.npy',  siglip_test)
else:
    siglip_train = np.load(cache / 'siglip_train.npy')
    siglip_test  = np.load(cache / 'siglip_test.npy')

print(f"SigLIP train: {siglip_train.shape}  test: {siglip_test.shape}")
# Update dim if different from expected
SIGLIP_DIM = siglip_train.shape[1]
print(f"SigLIP embedding dim: {SIGLIP_DIM}")

Loading SigLIP So400m-patch14-224...


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

SigLIP:   0%|          | 0/150 [00:00<?, ?it/s]

Loading SigLIP So400m-patch14-224...


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

SigLIP:   0%|          | 0/65 [00:00<?, ?it/s]

SigLIP train: (4800, 1152)  test: (2058, 1152)
SigLIP embedding dim: 1152


In [6]:
# ============================================================
# CELL 6: FEATURE EXTRACTION — DINOv2-Large @ 518px (1024-dim)
# ============================================================
# Upgraded from DINOv2-Base (768d) to Large (1024d).
# 518px resolution captures checkerboard artifacts.
set_seeds()

def extract_dino_large_features(image_paths, batch_size=16):
    print("Loading DINOv2-Large (vit_large_patch14_dinov2)...")
    model = timm.create_model('vit_large_patch14_dinov2',
                               pretrained=True, num_classes=0)
    model = model.to(DEVICE).eval()

    tfm = T.Compose([
        T.Resize(518, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(518),
        T.ToTensor(),
        T.Normalize(DINO_MEAN, DINO_STD),
    ])

    feats_all = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='DINOv2-L'):
        batch_imgs = []
        for p in image_paths[i:i+batch_size]:
            img = load_image_pil(p)
            if img is not None:
                batch_imgs.append(tfm(img))
            else:
                batch_imgs.append(torch.zeros(3, 518, 518))
        batch = torch.stack(batch_imgs).to(DEVICE)
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                feats = model(batch)
        feats = feats.float()
        feats = feats / feats.norm(dim=-1, keepdim=True)
        feats_all.append(feats.cpu().numpy())

    del model
    torch.cuda.empty_cache()
    gc.collect()
    return np.vstack(feats_all).astype(np.float32)

if FORCE_FRESH or not (cache / 'dino_l_train.npy').exists():
    dino_l_train = extract_dino_large_features(train_paths)
    dino_l_test  = extract_dino_large_features(test_paths)
    np.save(cache / 'dino_l_train.npy', dino_l_train)
    np.save(cache / 'dino_l_test.npy',  dino_l_test)
else:
    dino_l_train = np.load(cache / 'dino_l_train.npy')
    dino_l_test  = np.load(cache / 'dino_l_test.npy')

print(f"DINOv2-L train: {dino_l_train.shape}  test: {dino_l_test.shape}")
DINO_L_DIM = dino_l_train.shape[1]
print(f"DINOv2-L embedding dim: {DINO_L_DIM}")

Loading DINOv2-Large (vit_large_patch14_dinov2)...


DINOv2-L:   0%|          | 0/300 [00:00<?, ?it/s]

Loading DINOv2-Large (vit_large_patch14_dinov2)...


DINOv2-L:   0%|          | 0/129 [00:00<?, ?it/s]

DINOv2-L train: (4800, 1024)  test: (2058, 1024)
DINOv2-L embedding dim: 1024


In [7]:
# ============================================================
# CELL 7: FEATURE EXTRACTION — EfficientNet-B0 (1280-dim)
# ============================================================
set_seeds()

def extract_cnn_features(image_paths, batch_size=64):
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    model.classifier = nn.Identity()
    model = model.to(DEVICE).eval()

    tfm = T.Compose([
        T.Resize((224, 224)), T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    feats_all = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc='CNN'):
        batch_imgs = []
        for p in image_paths[i:i+batch_size]:
            img = load_image_pil(p)
            if img is not None:
                batch_imgs.append(tfm(img))
            else:
                batch_imgs.append(torch.zeros(3, 224, 224))
        batch = torch.stack(batch_imgs).to(DEVICE)
        with torch.no_grad():
            feats = model(batch)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        feats_all.append(feats.cpu().numpy())

    del model
    torch.cuda.empty_cache()
    gc.collect()
    return np.vstack(feats_all).astype(np.float32)

if FORCE_FRESH or not (cache / 'cnn_train.npy').exists():
    cnn_train = extract_cnn_features(train_paths)
    cnn_test  = extract_cnn_features(test_paths)
    np.save(cache / 'cnn_train.npy', cnn_train)
    np.save(cache / 'cnn_test.npy',  cnn_test)
else:
    cnn_train = np.load(cache / 'cnn_train.npy')
    cnn_test  = np.load(cache / 'cnn_test.npy')

print(f"CNN train: {cnn_train.shape}  test: {cnn_test.shape}")

CNN:   0%|          | 0/75 [00:00<?, ?it/s]

CNN:   0%|          | 0/33 [00:00<?, ?it/s]

CNN train: (4800, 1280)  test: (2058, 1280)


In [8]:
# ============================================================
# CELL 8: DIRE — Diffusion Iterative Reconstruction Error (FIXED)
# ============================================================
import os
import json
import torch
import torch.nn.functional as F
from PIL import Image
import numpy as np
from safetensors.torch import load_file as load_safetensors
from tqdm import tqdm
import gc

set_seeds()

DIRE_DIM = 20

def load_sd_unet():
    from huggingface_hub import hf_hub_download
    from diffusers import UNet2DConditionModel
    
    # Download UNet config and weights
    config_path = hf_hub_download("Manojb/stable-diffusion-2-1-base", "unet/config.json")
    weight_path = hf_hub_download("Manojb/stable-diffusion-2-1-base",
                                   "unet/diffusion_pytorch_model.safetensors")

    with open(config_path, 'r') as f:
        config = json.load(f)

    # Build UNet 
    try:
        unet = UNet2DConditionModel.from_pretrained("Manojb/stable-diffusion-2-1-base",
                                                      subfolder="unet")
    except Exception:
        unet = UNet2DConditionModel(**config)
        state_dict = load_safetensors(weight_path)
        unet.load_state_dict(state_dict)

    unet = unet.to(DEVICE).eval().half()
    return unet

def compute_dire_features_single(img_pil, unet, size=512):
    # Ensure image is strictly 3-channel RGB
    img = img_pil.convert('RGB').resize((size, size), Image.Resampling.LANCZOS)
    img_np = np.array(img).astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0)
    img_tensor = (img_tensor * 2.0 - 1.0).to(DEVICE).half()

    # Encode to latent-like space -> Shape [1, 3, 64, 64]
    latent = F.interpolate(img_tensor, size=(64, 64), mode='bilinear', align_corners=False)
    
    # FIX: SD UNet expects 4 channels. Pad the 3 RGB channels with a 4th zero channel.
    zeros_channel = torch.zeros((1, 1, 64, 64), device=DEVICE, dtype=torch.float16)
    latent = torch.cat([latent, zeros_channel], dim=1) 

    features = []
    # Test at multiple noise levels
    for noise_level in [0.02, 0.1, 0.3]:
        noise = torch.randn_like(latent) * noise_level
        noisy_latent = latent + noise

        timestep = torch.tensor([int(noise_level * 1000)], device=DEVICE)
        dummy_encoder = torch.zeros(1, 77, 1024, device=DEVICE, dtype=torch.float16)

        with torch.no_grad():
            with torch.cuda.amp.autocast():
                pred = unet(noisy_latent, timestep, encoder_hidden_states=dummy_encoder).sample

        # Reconstruction error (Drop the 4th dummy channel)
        error = (pred[:, :3, :, :].float() - latent[:, :3, :, :].float()).abs()
        error_np = error.cpu().numpy().flatten()

        features.extend([
            np.mean(error_np),
            np.std(error_np),
            np.percentile(error_np, 5),
            np.percentile(error_np, 50),
            np.percentile(error_np, 95),
            np.percentile(error_np, 99),
            float(np.mean(error_np > 0.1)),
        ])

    features = features[:DIRE_DIM]
    while len(features) < DIRE_DIM:
        features.append(0.0)
    return np.array(features, dtype=np.float32)

def extract_dire_features(image_paths, batch_size=1):
    print("Loading Stable Diffusion UNet for DIRE...")
    try:
        unet = load_sd_unet()
    except Exception as e:
        print(f"DIRE FAILED to load UNet: {e}")
        return np.zeros((len(image_paths), DIRE_DIM), dtype=np.float32)

    print(f"UNet loaded. Extracting DIRE features for {len(image_paths)} images...")
    feats_all = []
    failed = 0
    for i, p in enumerate(tqdm(image_paths, desc='DIRE')):
        img = load_image_pil(p) 
        if img is not None:
            try:
                feat = compute_dire_features_single(img, unet)
            except Exception as e:
                if failed == 0: 
                    print(f"\n[DEBUG] Extraction failed on image {i}. Error: {e}")
                feat = np.zeros(DIRE_DIM, dtype=np.float32)
                failed += 1
        else:
            feat = np.zeros(DIRE_DIM, dtype=np.float32)
            failed += 1
        feats_all.append(feat)

        if (i + 1) % 500 == 0:
            torch.cuda.empty_cache()

    del unet
    torch.cuda.empty_cache()
    gc.collect()

    if failed > 0:
        print(f"  WARNING: {failed}/{len(image_paths)} images failed DIRE extraction")
    return np.vstack(feats_all).astype(np.float32)

# ── Extract & Cache ──────────────────────────────────────────
if FORCE_FRESH or not (cache / 'dire_train.npy').exists():
    dire_train = extract_dire_features(train_paths)
    dire_test  = extract_dire_features(test_paths)
    np.save(cache / 'dire_train.npy', dire_train)
    np.save(cache / 'dire_test.npy',  dire_test)
else:
    dire_train = np.load(cache / 'dire_train.npy')
    dire_test  = np.load(cache / 'dire_test.npy')

print(f"DIRE train: {dire_train.shape}  test: {dire_test.shape}")

dire_alive = (dire_train.std(axis=0) > 1e-10).sum()
print(f"DIRE alive features: {dire_alive}/{dire_train.shape[1]}")
DIRE_ENABLED = dire_alive > 5
print(f"DIRE enabled for ensemble: {DIRE_ENABLED}")

Loading Stable Diffusion UNet for DIRE...
UNet loaded. Extracting DIRE features for 4800 images...


DIRE: 100%|██████████| 4800/4800 [17:05<00:00,  4.68it/s]


Loading Stable Diffusion UNet for DIRE...
UNet loaded. Extracting DIRE features for 2058 images...


DIRE: 100%|██████████| 2058/2058 [07:20<00:00,  4.67it/s]


DIRE train: (4800, 20)  test: (2058, 20)
DIRE alive features: 20/20
DIRE enabled for ensemble: True


In [9]:
# ============================================================
# CELL 9: FORENSIC FEATURE EXTRACTION (ELA + FFT + Noise ~114d)
# ============================================================
# Same as v5 — proven working code
from scipy.ndimage import median_filter
set_seeds()

FORENSIC_DIM = 114

def extract_ela_features(img_np):
    features = []
    img_pil = Image.fromarray(img_np.astype(np.uint8))
    for quality in [90, 75, 50]:
        buffer = io.BytesIO()
        img_pil.save(buffer, 'JPEG', quality=quality)
        buffer.seek(0)
        recompressed = np.array(Image.open(buffer), dtype=np.float64)
        ela = np.abs(img_np.astype(np.float64) - recompressed)
        for c in range(3):
            ch = ela[:, :, c]
            features.extend([np.mean(ch), np.std(ch)])
        ela_gray = np.mean(ela, axis=2)
        features.extend([np.percentile(ela_gray, 95), np.percentile(ela_gray, 5)])
    return np.array(features, dtype=np.float32)

def extract_fft_features(img_np):
    gray = np.mean(img_np, axis=2)
    f_transform = fft2(gray)
    f_shift = fftshift(f_transform)
    magnitude = np.log1p(np.abs(f_shift))
    h, w = magnitude.shape
    cy, cx = h // 2, w // 2
    max_radius = min(cy, cx)
    n_bins = 30
    radial_profile = np.zeros(n_bins)
    for i in range(n_bins):
        r_inner = int(i * max_radius / n_bins)
        r_outer = int((i + 1) * max_radius / n_bins)
        y, x = np.ogrid[-cy:h-cy, -cx:w-cx]
        mask = (x*x + y*y >= r_inner**2) & (x*x + y*y < r_outer**2)
        if mask.any():
            radial_profile[i] = np.mean(magnitude[mask])
    features = list(radial_profile)
    features.extend([np.mean(magnitude), np.std(magnitude),
                     np.sum(magnitude[cy-10:cy+10, cx-10:cx+10]),
                     np.sum(magnitude) - np.sum(magnitude[cy-10:cy+10, cx-10:cx+10])])
    y_grid, x_grid = np.ogrid[-cy:h-cy, -cx:w-cx]
    r_sq = y_grid**2 + x_grid**2
    low_e = np.sum(magnitude[r_sq < (max_radius * 0.2)**2])
    mid_e = np.sum(magnitude[(r_sq >= (max_radius * 0.2)**2) & (r_sq < (max_radius * 0.5)**2)])
    high_e = np.sum(magnitude[r_sq >= (max_radius * 0.5)**2])
    total_e = low_e + mid_e + high_e + 1e-10
    features.extend([low_e/total_e, mid_e/total_e, high_e/total_e, high_e/(low_e+1e-10)])
    valid = radial_profile > 0
    slope = np.polyfit(np.log(np.arange(1, n_bins+1)[valid]), np.log(radial_profile[valid]), 1)[0] if valid.sum() > 5 else 0.0
    features.append(slope)
    phase = np.angle(f_shift)
    features.extend([np.mean(phase), np.std(phase),
                     np.mean(np.abs(np.diff(phase, axis=0))),
                     np.mean(np.abs(np.diff(phase, axis=1)))])
    return np.array(features[:50], dtype=np.float32)

def extract_noise_features(img_np):
    gray = np.mean(img_np, axis=2)
    features = []
    for wavelet in ['db1', 'db2']:
        coeffs = pywt.dwt2(gray, wavelet)
        cA, (cH, cV, cD) = coeffs
        for detail in [cH, cV, cD]:
            features.extend([np.mean(np.abs(detail)), np.std(detail),
                             np.percentile(np.abs(detail), 99), np.mean(detail**2)])
    denoised = median_filter(gray, size=3)
    noise = gray - denoised
    features.extend([np.mean(noise), np.std(noise), np.mean(noise**2),
                     np.percentile(noise, 1), np.percentile(noise, 99)])
    return np.array(features[:40], dtype=np.float32)

def extract_forensic_single(path):
    img_np = load_image_np(path, size=(256, 256))
    if img_np is None:
        return np.zeros(FORENSIC_DIM, dtype=np.float32)
    ela  = extract_ela_features(img_np)
    fft  = extract_fft_features(img_np)
    noise = extract_noise_features(img_np)
    combined = np.concatenate([ela, fft, noise])
    if len(combined) > FORENSIC_DIM:
        combined = combined[:FORENSIC_DIM]
    elif len(combined) < FORENSIC_DIM:
        combined = np.pad(combined, (0, FORENSIC_DIM - len(combined)))
    return combined.astype(np.float32)

def extract_forensic_batch(paths):
    return np.vstack([extract_forensic_single(p) for p in tqdm(paths, desc='Forensic')])

if FORCE_FRESH or not (cache / 'forensic_train.npy').exists():
    forensic_train = extract_forensic_batch(train_paths)
    forensic_test  = extract_forensic_batch(test_paths)
    np.save(cache / 'forensic_train.npy', forensic_train)
    np.save(cache / 'forensic_test.npy',  forensic_test)
else:
    forensic_train = np.load(cache / 'forensic_train.npy')
    forensic_test  = np.load(cache / 'forensic_test.npy')

print(f"Forensic train: {forensic_train.shape}  test: {forensic_test.shape}")
alive = (forensic_train.std(axis=0) > 1e-10).sum()
print(f"Alive: {alive}/{forensic_train.shape[1]}")

Forensic: 100%|██████████| 2058/2058 [01:34<00:00, 21.87it/s]

Forensic train: (4800, 114)  test: (2058, 114)
Alive: 96/114


In [10]:
# ============================================================
# CELL 10: SHARED CV INFRASTRUCTURE (from v5, proven working)
# ============================================================
results_tracker = {}
oof_store       = {}
test_pred_store = {}

def evaluate_cv(name, X, y, model_factory, n_splits=N_FOLDS, store_oof=True):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y))
    tr_f1s, val_f1s, aucs = [], [], []
    print(f"\n{'='*62}\n  {name}\n{'='*62}")
    print(f"{'Fold':<5} {'Tr-F1':<8} {'Va-F1':<8} {'Gap':<7} {'AUC':<8} Status")
    print('-'*48)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        Xtr, Xv = X[tr_idx], X[val_idx]
        ytr, yv = y[tr_idx], y[val_idx]
        m = model_factory()
        m.fit(Xtr, ytr)
        tp = m.predict_proba(Xtr)[:, 1]
        vp = m.predict_proba(Xv)[:, 1]
        tf1 = f1_score(ytr, (tp >= 0.5).astype(int))
        vf1 = f1_score(yv,  (vp >= 0.5).astype(int))
        au  = roc_auc_score(yv, vp)
        gap = tf1 - vf1
        oof[val_idx] = vp
        tr_f1s.append(tf1); val_f1s.append(vf1); aucs.append(au)
        st = 'PASS' if gap < 0.08 else ('WARN' if gap < 0.10 else 'FAIL')
        print(f"{fold+1:<5} {tf1:<8.4f} {vf1:<8.4f} {gap:<7.4f} {au:<8.4f} {st}")
    mv = np.mean(val_f1s); sv = np.std(val_f1s)
    mt = np.mean(tr_f1s); ma = np.mean(aucs); mg = mt - mv
    lb = 'PASS' if mg < 0.08 else ('WARN' if mg < 0.10 else 'FAIL')
    print('-'*48)
    print(f"MEAN  {mt:<8.4f} {mv:<8.4f} {mg:<7.4f} {ma:<8.4f} [{lb}]")
    print(f"STD            {sv:<8.4f}")
    return {'name': name, 'val_f1_mean': mv, 'val_f1_std': sv,
            'train_f1_mean': mt, 'gap': mg, 'val_auc_mean': ma,
            'oof_proba': oof.copy() if store_oof else None}

def run_epoch(model, loader, optimizer, scaler, criterion, is_train, use_mixup=False):
    model.train() if is_train else model.eval()
    tot_loss = 0.0; preds = []; trues = []
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
                if is_train and use_mixup and random.random() < 0.5:
                    imgs, y_a, y_b, lam = mixup_data(imgs, labels, alpha=0.3)
                    logits = model(imgs)
                    loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
                else:
                    logits = model(imgs)
                    loss = criterion(logits, labels.float())
            if is_train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            tot_loss += loss.item() * len(labels)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            preds.extend(probs.tolist())
            trues.extend(labels.cpu().numpy().tolist())
    f1_val = f1_score(trues, (np.array(preds) >= 0.5).astype(int), zero_division=0)
    return tot_loss / len(trues), f1_val, np.array(preds)

def train_finetune_fold(model, tr_ds, val_ds, criterion,
                        freeze_fn, unfreeze_fn, get_opt_fn,
                        fold_num, batch_size=32):
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                       num_workers=4, pin_memory=True, drop_last=True)
    va_ld = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                       num_workers=4, pin_memory=True)
    freeze_fn(model)
    opt1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=1e-3, weight_decay=0.01)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=10, eta_min=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))
    best_f1 = 0; best_state = None; best_vp = None
    patience = 5; no_improve = 0
    for ep in range(1, 11):
        tl, tf, _ = run_epoch(model, tr_ld, opt1, scaler, criterion, True, use_mixup=True)
        vl, vf, vp = run_epoch(model, va_ld, None, scaler, criterion, False)
        sch1.step()
        if vf > best_f1:
            best_f1 = vf; no_improve = 0
            best_state = copy.deepcopy(model.state_dict()); best_vp = vp.copy()
        else:
            no_improve += 1
        if no_improve >= patience: break
    s1_f1 = best_f1
    model.load_state_dict(best_state)
    unfreeze_fn(model)
    opt2 = get_opt_fn(model)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=15, eta_min=1e-7)
    no_improve = 0
    for ep in range(1, 16):
        tl, tf, _ = run_epoch(model, tr_ld, opt2, scaler, criterion, True, use_mixup=True)
        vl, vf, vp = run_epoch(model, va_ld, None, scaler, criterion, False)
        sch2.step()
        gap = tf - vf
        if vf > best_f1:
            best_f1 = vf; no_improve = 0
            best_state = copy.deepcopy(model.state_dict()); best_vp = vp.copy()
        else:
            no_improve += 1
        if no_improve >= patience: break
        if gap > 0.12:
            print(f"   Fold {fold_num}: gap {gap:.4f} > 0.12, early stop")
            break
    model.load_state_dict(best_state)
    print(f"   Fold {fold_num}: Stage1 F1={s1_f1:.4f} -> Stage2 F1={best_f1:.4f}")
    return best_f1, best_vp, best_state

print("CV infrastructure ready for RTX 3090.")

CV infrastructure ready for RTX 3090.


In [16]:
# CLEANUP: Free GPU memory from previous cells
import torch, gc
torch.cuda.empty_cache()
gc.collect()
print(f"Free VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

Free VRAM: 21.8 GB


In [17]:
# ============================================================
# CELL 11: SigLIP So400m FINE-TUNE — 5-Fold CV (OOM-Safe)
# ============================================================
from transformers import SiglipModel
set_seeds()

class SigLIPFineTuner(nn.Module):
    def __init__(self, vision_model, embed_dim):
        super().__init__()
        self.visual = vision_model
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        outputs = self.visual(pixel_values=x)
        f = outputs.pooler_output.float()
        return self.head(f).squeeze(-1)

def freeze_siglip(model):
    for p in model.visual.parameters():
        p.requires_grad = False

def unfreeze_siglip_blocks(model, n=2):
    layers = model.visual.encoder.layers
    for p in layers[-n:].parameters():
        p.requires_grad = True
    if hasattr(model.visual, 'post_layernorm'):
        for p in model.visual.post_layernorm.parameters():
            p.requires_grad = True

def get_siglip_optimizer(model, backbone_lr=5e-6, head_lr=1e-4):
    backbone_params = [p for n, p in model.visual.named_parameters() if p.requires_grad]
    head_params = list(model.head.parameters())
    return torch.optim.AdamW([
        {'params': backbone_params, 'lr': backbone_lr, 'weight_decay': 0.05},
        {'params': head_params,     'lr': head_lr,     'weight_decay': 0.01}
    ])

# ── OOM-safe checkpoint: save state_dict to CPU ─────────────
def save_state_cpu(model):
    return {k: v.cpu().clone() for k, v in model.state_dict().items()}

# ── OOM-safe train_finetune_fold (replaces the one in Cell 10) ──
def train_finetune_fold_safe(model, tr_ds, val_ds, criterion,
                              freeze_fn, unfreeze_fn, get_opt_fn,
                              fold_num, batch_size=16):
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                       num_workers=4, pin_memory=True, drop_last=True)
    va_ld = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                       num_workers=4, pin_memory=True)

    # Stage 1: Head only
    freeze_fn(model)
    opt1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=1e-3, weight_decay=0.01)
    sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=10, eta_min=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    best_f1 = 0; best_state = None; best_vp = None
    patience = 5; no_improve = 0

    for ep in range(1, 11):
        tl, tf, _ = run_epoch(model, tr_ld, opt1, scaler, criterion, True, use_mixup=True)
        vl, vf, vp = run_epoch(model, va_ld, None, scaler, criterion, False)
        sch1.step()
        if vf > best_f1:
            best_f1 = vf; no_improve = 0
            best_state = save_state_cpu(model)
            best_vp = vp.copy()
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    s1_f1 = best_f1
    model.load_state_dict(best_state)

    # Stage 2: Unfreeze last blocks
    unfreeze_fn(model)
    opt2 = get_opt_fn(model)
    sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=15, eta_min=1e-7)
    no_improve = 0

    for ep in range(1, 16):
        tl, tf, _ = run_epoch(model, tr_ld, opt2, scaler, criterion, True, use_mixup=True)
        vl, vf, vp = run_epoch(model, va_ld, None, scaler, criterion, False)
        sch2.step()
        gap = tf - vf
        if vf > best_f1:
            best_f1 = vf; no_improve = 0
            best_state = save_state_cpu(model)
            best_vp = vp.copy()
        else:
            no_improve += 1
        if no_improve >= patience:
            break
        if gap > 0.12:
            print(f"   Fold {fold_num}: gap {gap:.4f} > 0.12, early stop")
            break

    model.load_state_dict(best_state)
    print(f"   Fold {fold_num}: Stage1 F1={s1_f1:.4f} -> Stage2 F1={best_f1:.4f}")
    return best_f1, best_vp, best_state

# ── Transforms ───────────────────────────────────────────────
siglip_train_tfm = get_train_transform_albu(SIGLIP_MEAN, SIGLIP_STD, size=224)
siglip_val_tfm   = get_val_transform_albu(SIGLIP_MEAN, SIGLIP_STD, size=224)
siglip_tta_tfm   = get_tta_transform_albu(SIGLIP_MEAN, SIGLIP_STD, size=224)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
siglip_oof = np.zeros(len(y_all))
siglip_fold_f1s = []
siglip_fold_models = []

print("=" * 60)
print(f"SigLIP So400m FINE-TUNE on {torch.cuda.get_device_name(0)}")
print("=" * 60)

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_paths, y_all)):
    print(f"\n-- Fold {fold+1}/{N_FOLDS} --")
    tr_paths_f  = [train_paths[i] for i in tr_idx]
    val_paths_f = [train_paths[i] for i in val_idx]
    tr_labels_f  = y_all[tr_idx].tolist()
    val_labels_f = y_all[val_idx].tolist()

    tr_ds  = AlbuDataset(tr_paths_f, tr_labels_f, siglip_train_tfm)
    val_ds = AlbuDataset(val_paths_f, val_labels_f, siglip_val_tfm)

    base_model = SiglipModel.from_pretrained("google/siglip-so400m-patch14-224")
    model = SigLIPFineTuner(base_model.vision_model, embed_dim=SIGLIP_DIM).to(DEVICE)
    del base_model
    torch.cuda.empty_cache()
    gc.collect()

    n_pos = sum(tr_labels_f); n_neg = len(tr_labels_f) - n_pos
    pw = torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    fold_f1, fold_vp, fold_state = train_finetune_fold_safe(
        model, tr_ds, val_ds, criterion,
        freeze_fn=freeze_siglip,
        unfreeze_fn=lambda m: unfreeze_siglip_blocks(m, n=2),
        get_opt_fn=get_siglip_optimizer,
        fold_num=fold+1, batch_size=16
    )

    siglip_oof[val_idx] = fold_vp
    siglip_fold_f1s.append(fold_f1)
    siglip_fold_models.append(fold_state)  # Already on CPU

    del model
    torch.cuda.empty_cache()
    gc.collect()

# ── Test inference ──────────────────────────────────────────
print(f"\n-- Test Inference ({N_FOLDS} models x {N_TTA} TTA) --")
siglip_test_proba = np.zeros(len(test_paths))

for fold, fold_state in enumerate(siglip_fold_models):
    print(f"   Inference Fold {fold+1}...")
    base_model = SiglipModel.from_pretrained("google/siglip-so400m-patch14-224")
    model = SigLIPFineTuner(base_model.vision_model, embed_dim=SIGLIP_DIM).to(DEVICE)
    model.load_state_dict(fold_state)
    model.eval()
    del base_model
    torch.cuda.empty_cache()

    test_ds = AlbuDataset(test_paths, [-1]*len(test_paths), siglip_tta_tfm)
    test_ld = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

    fold_tta = np.zeros(len(test_paths))
    for t in range(N_TTA):
        preds = []
        with torch.no_grad():
            for imgs, _ in test_ld:
                with torch.cuda.amp.autocast():
                    logits = model(imgs.to(DEVICE))
                preds.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        fold_tta += np.array(preds)
    siglip_test_proba += (fold_tta / N_TTA)

    del model
    torch.cuda.empty_cache()
    gc.collect()

siglip_test_proba /= N_FOLDS
siglip_oof_auc = roc_auc_score(y_all, siglip_oof)
results_tracker['siglip_finetune'] = {
    'name': 'SigLIP-FT (5-fold)', 'val_f1_mean': np.mean(siglip_fold_f1s),
    'val_auc_mean': siglip_oof_auc, 'oof_proba': siglip_oof.copy()
}
oof_store['siglip_finetune'] = siglip_oof.copy()
test_pred_store['siglip_finetune'] = siglip_test_proba.copy()
print(f"\nSigLIP Fine-Tune: Val F1={np.mean(siglip_fold_f1s):.4f}, OOF AUC={siglip_oof_auc:.4f}")

SigLIP So400m FINE-TUNE on NVIDIA GeForce RTX 3090

-- Fold 1/5 --


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

   Fold 1: Stage1 F1=0.8703 -> Stage2 F1=0.8912

-- Fold 2/5 --


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

   Fold 2: Stage1 F1=0.8825 -> Stage2 F1=0.9059

-- Fold 3/5 --


...

   Inference Fold 4...


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

   Inference Fold 5...


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]


SigLIP Fine-Tune: Val F1=0.9002, OOF AUC=0.9679


In [18]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"Free VRAM: {free:.1f} GB")

Free VRAM: 25.4 GB


In [19]:
# ============================================================
# CELL 12: DINOv2-Large FINE-TUNE — 5-Fold CV @ 518px
# ============================================================
set_seeds()

class DINOv2LFineTuner(nn.Module):
    def __init__(self, backbone, embed_dim=1024):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        f = self.backbone(x)
        return self.head(f).squeeze(-1)

def freeze_dinol(model):
    for p in model.backbone.parameters():
        p.requires_grad = False

def unfreeze_dinol_blocks(model, n=3):
    blocks = model.backbone.blocks
    for p in blocks[-n:].parameters():
        p.requires_grad = True
    if hasattr(model.backbone, 'norm'):
        for p in model.backbone.norm.parameters():
            p.requires_grad = True

def get_dinol_optimizer(model, backbone_lr=5e-6, head_lr=1e-4):
    backbone_params = [p for n, p in model.backbone.named_parameters() if p.requires_grad]
    head_params = list(model.head.parameters())
    return torch.optim.AdamW([
        {'params': backbone_params, 'lr': backbone_lr, 'weight_decay': 0.05},
        {'params': head_params,     'lr': head_lr,     'weight_decay': 0.01}
    ])

# DINOv2-L specific transforms (518x518)
dinol_train_tfm = A.Compose([
    A.RandomResizedCrop(size=(518, 518), scale=(0.8, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.GaussNoise(var_limit=(5, 30), p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.ImageCompression(quality_lower=70, quality_upper=100, p=0.3),
    A.Normalize(mean=DINO_MEAN, std=DINO_STD),
    A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.2),
])
dinol_val_tfm = A.Compose([
    A.Resize(height=518, width=518),
    A.Normalize(mean=DINO_MEAN, std=DINO_STD),
])
dinol_tta_tfm = A.Compose([
    A.RandomResizedCrop(size=(518, 518), scale=(0.9, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=DINO_MEAN, std=DINO_STD),
])

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
dinol_oof = np.zeros(len(y_all))
dinol_fold_f1s = []
dinol_fold_models = []

print("=" * 60)
print(f"DINOv2-Large @ 518px FINE-TUNE on {torch.cuda.get_device_name(0)}")
print("=" * 60)

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_paths, y_all)):
    print(f"\n-- Fold {fold+1}/{N_FOLDS} --")
    tr_paths_f = [train_paths[i] for i in tr_idx]
    val_paths_f = [train_paths[i] for i in val_idx]
    tr_labels_f = y_all[tr_idx].tolist()
    val_labels_f = y_all[val_idx].tolist()

    tr_ds  = AlbuDataset(tr_paths_f, tr_labels_f, dinol_train_tfm)
    val_ds = AlbuDataset(val_paths_f, val_labels_f, dinol_val_tfm)

    backbone = timm.create_model('vit_large_patch14_dinov2', pretrained=True, num_classes=0)
    model = DINOv2LFineTuner(backbone, embed_dim=DINO_L_DIM).to(DEVICE)

    n_pos = sum(tr_labels_f); n_neg = len(tr_labels_f) - n_pos
    pw = torch.tensor([n_neg / max(n_pos, 1)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    # batch_size=8 for 518px on RTX 3090 (safe for 24GB VRAM)
    fold_f1, fold_vp, fold_state = train_finetune_fold(
        model, tr_ds, val_ds, criterion,
        freeze_fn=freeze_dinol,
        unfreeze_fn=lambda m: unfreeze_dinol_blocks(m, n=3),
        get_opt_fn=get_dinol_optimizer,
        fold_num=fold+1, batch_size=8
    )

    dinol_oof[val_idx] = fold_vp
    dinol_fold_f1s.append(fold_f1)
    dinol_fold_models.append(fold_state)
    del model, backbone; torch.cuda.empty_cache(); gc.collect()

# ── Test inference ──────────────────────────────────────────
print(f"\n-- Test Inference ({N_FOLDS} models x {N_TTA} TTA) --")
dinol_test_proba = np.zeros(len(test_paths))

for fold, fold_state in enumerate(dinol_fold_models):
    print(f"   Inference Fold {fold+1}...")
    backbone = timm.create_model('vit_large_patch14_dinov2', pretrained=True, num_classes=0)
    model = DINOv2LFineTuner(backbone, embed_dim=DINO_L_DIM).to(DEVICE)
    model.load_state_dict(fold_state)
    model.eval()

    test_ds = AlbuDataset(test_paths, [-1]*len(test_paths), dinol_tta_tfm)
    test_ld = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)

    fold_tta = np.zeros(len(test_paths))
    for t in range(N_TTA):
        preds = []
        with torch.no_grad():
            for imgs, _ in test_ld:
                with torch.cuda.amp.autocast():
                    logits = model(imgs.to(DEVICE))
                preds.extend(torch.sigmoid(logits).cpu().numpy().tolist())
        fold_tta += np.array(preds)
    dinol_test_proba += (fold_tta / N_TTA)
    del model, backbone; torch.cuda.empty_cache(); gc.collect()

dinol_test_proba /= N_FOLDS
dinol_oof_auc = roc_auc_score(y_all, dinol_oof)
results_tracker['dinol_finetune'] = {
    'name': 'DINOv2-L-FT (5-fold)', 'val_f1_mean': np.mean(dinol_fold_f1s),
    'val_auc_mean': dinol_oof_auc, 'oof_proba': dinol_oof.copy()
}
oof_store['dinol_finetune'] = dinol_oof.copy()
test_pred_store['dinol_finetune'] = dinol_test_proba.copy()
print(f"\nDINOv2-L Fine-Tune: Val F1={np.mean(dinol_fold_f1s):.4f}, OOF AUC={dinol_oof_auc:.4f}")

DINOv2-Large @ 518px FINE-TUNE on NVIDIA GeForce RTX 3090

-- Fold 1/5 --
   Fold 1: Stage1 F1=0.7823 -> Stage2 F1=0.8509

-- Fold 2/5 --
   Fold 2: Stage1 F1=0.8024 -> Stage2 F1=0.8657

-- Fold 3/5 --
   Fold 3: Stage1 F1=0.7804 -> Stage2 F1=0.8545

-- Fold 4/5 --
   Fold 4: Stage1 F1=0.8109 -> Stage2 F1=0.8831

-- Fold 5/5 --
   Fold 5: Stage1 F1=0.7824 -> Stage2 F1=0.8535

-- Test Inference (5 models x 5 TTA) --
   Inference Fold 1...
   Inference Fold 2...
   Inference Fold 3...
   Inference Fold 4...
   Inference Fold 5...

DINOv2-L Fine-Tune: Val F1=0.8615, OOF AUC=0.9376


In [20]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"Free VRAM: {free:.1f} GB")

Free VRAM: 19.3 GB


In [21]:
# ============================================================
# CELL 13: DIRE + FORENSIC → XGBoost (5-Fold CV)
# ============================================================
set_seeds()

# Clean dead features
vt = VarianceThreshold(threshold=1e-10)
forensic_train_clean = vt.fit_transform(forensic_train)
forensic_test_clean  = vt.transform(forensic_test)
print(f"Forensic: {forensic_train.shape[1]} -> {forensic_train_clean.shape[1]} features")

XGB_PARAMS = dict(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.7, colsample_bytree=0.7, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=1.0, gamma=0.1,
    use_label_encoder=False, eval_metric='logloss',
    random_state=SEED, tree_method='hist', device='cuda'
)

# ── DIRE-only model (if enabled) ─────────────────────────────
if DIRE_ENABLED:
    dire_clean = vt.fit_transform(dire_train) if dire_train.std(axis=0).min() < 1e-10 else dire_train
    dire_test_clean = vt.transform(dire_test) if dire_train.std(axis=0).min() < 1e-10 else dire_test

    res_dire = evaluate_cv('XGB-DIRE', dire_clean, y_all,
        lambda: Pipeline([('sc', RobustScaler()), ('xgb', xgb.XGBClassifier(**XGB_PARAMS))]))
    results_tracker['xgb_dire'] = res_dire
    oof_store['xgb_dire'] = res_dire['oof_proba']
    print(f"DIRE alone: Val F1={res_dire['val_f1_mean']:.4f}")
else:
    print("DIRE disabled — skipping DIRE-only model")

# ── Combined: DIRE + Forensic + CNN ──────────────────────────
components = [forensic_train_clean]
components_test = [forensic_test_clean]
combo_name = "Forensic"

if DIRE_ENABLED and results_tracker.get('xgb_dire', {}).get('val_f1_mean', 0) > 0.55:
    components.append(dire_clean)
    components_test.append(dire_test_clean)
    combo_name += "+DIRE"

components.append(cnn_train)
components_test.append(cnn_test)
combo_name += "+CNN"

X_combo = np.hstack(components)
X_combo_test = np.hstack(components_test)

res_combo = evaluate_cv(f'XGB-{combo_name}', X_combo, y_all,
    lambda: Pipeline([
        ('sc', RobustScaler()),
        ('pca', PCA(n_components=min(128, X_combo.shape[1]), random_state=SEED)),
        ('xgb', xgb.XGBClassifier(**XGB_PARAMS))
    ]))
results_tracker['xgb_combo'] = res_combo
oof_store['xgb_combo'] = res_combo['oof_proba']

# Train on full data for test preds
final_pipe = Pipeline([
    ('sc', RobustScaler()),
    ('pca', PCA(n_components=min(128, X_combo.shape[1]), random_state=SEED)),
    ('xgb', xgb.XGBClassifier(**XGB_PARAMS))
])
final_pipe.fit(X_combo, y_all)
test_pred_store['xgb_combo'] = final_pipe.predict_proba(X_combo_test)[:, 1]

print(f"\nBest XGB model: XGB-{combo_name}")
print(f"  Val F1: {res_combo['val_f1_mean']:.4f}")

Forensic: 114 -> 96 features

  XGB-DIRE
Fold  Tr-F1    Va-F1    Gap     AUC      Status
------------------------------------------------
1     0.8389   0.5364   0.3025  0.6356   FAIL
2     0.8366   0.5557   0.2810  0.6297   FAIL
3     0.8447   0.5749   0.2698  0.6467   FAIL
4     0.8363   0.5335   0.3028  0.6101   FAIL
5     0.8436   0.5783   0.2653  0.6617   FAIL
------------------------------------------------
MEAN  0.8400   0.5558   0.2843  0.6368   [FAIL]
STD            0.0187  
DIRE alone: Val F1=0.5558

  XGB-Forensic+DIRE+CNN
Fold  Tr-F1    Va-F1    Gap     AUC      Status
------------------------------------------------
1     0.9881   0.7474   0.2407  0.8317   FAIL
2     0.9873   0.7651   0.2222  0.8574   FAIL
3     0.9849   0.7749   0.2100  0.8555   FAIL
4     0.9884   0.7577   0.2307  0.8440   FAIL
5     0.9898   0.7692   0.2205  0.8350   FAIL
------------------------------------------------
MEAN  0.9877   0.7629   0.2248  0.8447   [FAIL]
STD            0.0095  

Best XGB mo

In [22]:
import torch, gc
torch.cuda.empty_cache()
gc.collect()
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"Free VRAM: {free:.1f} GB")

Free VRAM: 19.3 GB


In [23]:
# ============================================================
# CELL 14: MULTI-MODEL ENSEMBLE + THRESHOLD TUNING
# ============================================================
set_seeds()

ensemble_keys = ['siglip_finetune', 'dinol_finetune']
# Add XGB combo if it adds signal (F1 > 0.60)
if results_tracker.get('xgb_combo', {}).get('val_f1_mean', 0) > 0.60:
    ensemble_keys.append('xgb_combo')

print("=" * 60)
print("ENSEMBLE COMPOSITION")
print("=" * 60)
for k in ensemble_keys:
    r = results_tracker[k]
    print(f"  {r['name']:<30} | Val F1: {r['val_f1_mean']:.4f}")

# ── Strategy A: Weighted Average ─────────────────────────────
print("\n-- Strategy A: Weighted Average --")
f1_sq = {k: results_tracker[k]['val_f1_mean']**2 for k in ensemble_keys}
total_w = sum(f1_sq.values())
weights = {k: v/total_w for k, v in f1_sq.items()}
for k, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"  {k:<30} w={w:.4f}")

oof_blend_A = sum(weights[k] * oof_store[k] for k in ensemble_keys)
best_thr_A = 0.5; best_f1_A = 0
for t in np.arange(0.30, 0.71, 0.01):
    f = f1_score(y_all, (oof_blend_A >= t).astype(int))
    if f > best_f1_A: best_f1_A = f; best_thr_A = round(t, 2)
print(f"Result A: OOF F1={best_f1_A:.4f} @ Thr={best_thr_A}")

# ── Strategy B: LogReg Meta-Learner ──────────────────────────
print("\n-- Strategy B: LogReg Meta-Learner --")
X_meta = np.column_stack([oof_store[k] for k in ensemble_keys])
skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
meta_oof = np.zeros(len(y_all))
for fold, (tr_idx, val_idx) in enumerate(skf_meta.split(X_meta, y_all)):
    lr = LogisticRegression(C=1.0, random_state=SEED, max_iter=1000)
    lr.fit(X_meta[tr_idx], y_all[tr_idx])
    meta_oof[val_idx] = lr.predict_proba(X_meta[val_idx])[:, 1]

best_thr_B = 0.5; best_f1_B = 0
for t in np.arange(0.30, 0.71, 0.01):
    f = f1_score(y_all, (meta_oof >= t).astype(int))
    if f > best_f1_B: best_f1_B = f; best_thr_B = round(t, 2)
print(f"Result B: OOF F1={best_f1_B:.4f} @ Thr={best_thr_B}")

# ── Select winner ────────────────────────────────────────────
print("\n" + "=" * 60)
if best_f1_B >= best_f1_A:
    print(f"WINNER: Meta-Learner ({best_f1_B:.4f})")
    BEST_THRESHOLD = best_thr_B; ensemble_method = 'meta'
    meta_final = LogisticRegression(C=1.0, random_state=SEED, max_iter=1000)
    meta_final.fit(X_meta, y_all)
    X_meta_test = np.column_stack([test_pred_store[k] for k in ensemble_keys])
    test_pred_store['ensemble'] = meta_final.predict_proba(X_meta_test)[:, 1]
    best_ensemble_f1 = best_f1_B; final_oof = meta_oof
else:
    print(f"WINNER: Weighted Average ({best_f1_A:.4f})")
    BEST_THRESHOLD = best_thr_A; ensemble_method = 'weighted_avg'
    test_blend = sum(weights[k] * test_pred_store[k] for k in ensemble_keys)
    test_pred_store['ensemble'] = test_blend
    best_ensemble_f1 = best_f1_A; final_oof = oof_blend_A

results_tracker['ensemble'] = {
    'name': f'Ensemble ({ensemble_method})', 'val_f1_mean': best_ensemble_f1,
    'val_auc_mean': roc_auc_score(y_all, final_oof), 'oof_proba': final_oof.copy()
}
oof_store['ensemble'] = final_oof.copy()

print(f"\nFINAL ENSEMBLE: {ensemble_method} | F1={best_ensemble_f1:.4f} | Thr={BEST_THRESHOLD}")
print("=" * 60)

ENSEMBLE COMPOSITION
  SigLIP-FT (5-fold)             | Val F1: 0.9002
  DINOv2-L-FT (5-fold)           | Val F1: 0.8615
  XGB-Forensic+DIRE+CNN          | Val F1: 0.7629

-- Strategy A: Weighted Average --
  siglip_finetune                w=0.3796
  dinol_finetune                 w=0.3477
  xgb_combo                      w=0.2726
Result A: OOF F1=0.9029 @ Thr=0.51

-- Strategy B: LogReg Meta-Learner --
Result B: OOF F1=0.9130 @ Thr=0.54

WINNER: Meta-Learner (0.9130)

FINAL ENSEMBLE: meta | F1=0.9130 | Thr=0.54


In [24]:
# ============================================================
# CELL 15: ANALYSIS & VISUALIZATION
# ============================================================
print("=" * 75)
print(f"{'Model':<35} {'Va-F1':<10} {'AUC':<10}")
print("=" * 75)

display_order = ['siglip_finetune', 'dinol_finetune', 'xgb_combo', 'ensemble']
if 'xgb_dire' in results_tracker:
    display_order.insert(2, 'xgb_dire')

for k in display_order:
    if k not in results_tracker: continue
    r = results_tracker[k]
    print(f"  {r['name']:<33} {r['val_f1_mean']:<10.4f} {r['val_auc_mean']:<10.4f}")
print("=" * 75)

# ── Confusion Matrix ─────────────────────────────────────────
best_oof = oof_store.get('ensemble', dinol_oof)
oof_preds = (best_oof >= BEST_THRESHOLD).astype(int)
cm = confusion_matrix(y_all, oof_preds)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Real', 'AI'], yticklabels=['Real', 'AI'])
axes[0].set_title(f'Confusion Matrix (OOF, thr={BEST_THRESHOLD})')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

for k in ['siglip_finetune', 'dinol_finetune', 'ensemble']:
    if k not in oof_store: continue
    fpr, tpr, _ = roc_curve(y_all, oof_store[k])
    auc_val = roc_auc_score(y_all, oof_store[k])
    axes[1].plot(fpr, tpr, label=f"{k} (AUC={auc_val:.4f})")
axes[1].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[1].set_title('ROC Curves'); axes[1].legend(); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
plt.tight_layout(); plt.savefig('analysis_v7.png', dpi=150); plt.show()

# ── Correlation ──────────────────────────────────────────────
corr_keys = [k for k in ['siglip_finetune', 'dinol_finetune', 'xgb_combo'] if k in oof_store]
if len(corr_keys) > 1:
    corr_data = np.column_stack([oof_store[k] for k in corr_keys])
    corr = np.corrcoef(corr_data.T)
    print("\nModel Correlation:")
    print(f"{'':>20}", '  '.join(f"{k[:12]:>12}" for k in corr_keys))
    for i, k in enumerate(corr_keys):
        print(f"{k[:20]:>20}", '  '.join(f"{corr[i,j]:>12.4f}" for j in range(len(corr_keys))))

# ── Ablation ─────────────────────────────────────────────────
print("\n--- ABLATION ---")
for k in display_order:
    if k not in results_tracker: continue
    print(f"  {results_tracker[k]['name']:<35} {results_tracker[k]['val_f1_mean']:.4f}")

Model                               Va-F1      AUC       
  SigLIP-FT (5-fold)                0.9002     0.9679    
  DINOv2-L-FT (5-fold)              0.8615     0.9376    
  XGB-DIRE                          0.5558     0.6368    
  XGB-Forensic+DIRE+CNN             0.7629     0.8447    
  Ensemble (meta)                   0.9130     0.9692    

Model Correlation:
                     siglip_finet  dinol_finetu     xgb_combo
     siglip_finetune       1.0000        0.7980        0.6387
      dinol_finetune       0.7980        1.0000        0.6224
           xgb_combo       0.6387        0.6224        1.0000

--- ABLATION ---
  SigLIP-FT (5-fold)                  0.9002
  DINOv2-L-FT (5-fold)                0.8615
  XGB-DIRE                            0.5558
  XGB-Forensic+DIRE+CNN               0.7629
  Ensemble (meta)                     0.9130


In [25]:
# ============================================================
# CELL 16: FINAL SUBMISSION
# ============================================================
set_seeds()

# Select best
best_key = 'ensemble'
# Check if single model beats ensemble
for k in ['dinol_finetune', 'siglip_finetune']:
    if results_tracker.get(k, {}).get('val_f1_mean', 0) > results_tracker['ensemble']['val_f1_mean']:
        best_key = k
        print(f"NOTE: {k} ({results_tracker[k]['val_f1_mean']:.4f}) > ensemble. Using single model.")

test_proba = test_pred_store[best_key]
thr = BEST_THRESHOLD

print("=" * 60)
print(f"FINAL MODEL: {best_key}")
print(f"Val F1: {results_tracker[best_key]['val_f1_mean']:.4f}")
print(f"Threshold: {thr}")
print("=" * 60)

preds_binary = (test_proba >= thr).astype(int)
submission = pd.DataFrame({
    'image_id': df_test['image_id'].values,
    'ground_truth': preds_binary
})

assert submission.shape == (2058, 2), f"Wrong shape: {submission.shape}"
assert submission['ground_truth'].isin([0, 1]).all()
assert not submission.isnull().any().any()

n0 = (submission['ground_truth'] == 0).sum()
n1 = (submission['ground_truth'] == 1).sum()
print(f"\nSanity checks PASSED")
print(f"  Real (0): {n0} ({n0/len(submission):.1%})")
print(f"  AI   (1): {n1} ({n1/len(submission):.1%})")
print(f"\nFirst 5 rows:")
print(submission.head())

out_path = '/home/jovyan/work/data/submission.csv'
submission.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"\nFINAL: {best_key} | Val F1={results_tracker[best_key]['val_f1_mean']:.4f} | Thr={thr}")

FINAL MODEL: ensemble
Val F1: 0.9130
Threshold: 0.54

Sanity checks PASSED
  Real (0): 1092 (53.1%)
  AI   (1): 966 (46.9%)

First 5 rows:
                                   image_id  ground_truth
0  3ecf1af5-6a8f-416a-9b4c-df9f2e0a0a80.jpg             0
1  2789b3fe-a337-4dc2-b42c-8bccde1f68fb.jpg             0
2  01a342c6-c3fc-4b55-8c22-13c1a556ba87.jpg             0
3  ac784910-b461-498d-b3a8-50b1e4116b11.jpg             0
4  6dcd4df6-7447-4bcf-a29b-f7f53b4c3ed4.jpg             0

Saved: /home/jovyan/work/data/submission.csv

FINAL: ensemble | Val F1=0.9130 | Thr=0.54
